In [7]:
from lago.lago import LinkStream, lago_communities
import pandas as pd

filename = 'syn_data/p1.0_mu0.2_l5_1.csv'

#filename = 'data/CollegeMsg.csv'
df = pd.read_csv(filename, header=None, index_col=False,names=["source","destination","timestamp"], skiprows=1)
time_links = df.values.tolist()

my_linkstream = LinkStream(is_stream_graph=False)
my_linkstream.add_links(time_links)

print(f"The link stream consists of {my_linkstream.nb_edges} temporal edges (or time links) accross {my_linkstream.nb_nodes} nodes and {my_linkstream.network_duration} time steps, of which only {my_linkstream.nb_timesteps} contain activity.")

The link stream consists of 515 temporal edges (or time links) accross 50 nodes and 541 time steps, of which only 333 contain activity.


/var/folders/g1/wg9xf7ks75b49ljdm6l5zy280000gp/T/ipykernel_4714/2508178066.py:7: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(filename, header=None, index_col=False,names=["source","destination","timestamp"], skiprows=1)


In [8]:
import importlib
from lago.lago import LinkStream, lago_communities

my_linkstream = LinkStream(is_stream_graph=True)
my_linkstream.add_links(time_links)

## Compute dynamic communities
dynamic_communities = lago_communities(
    my_linkstream,
    nb_iter=1,
    )

# Each dynamic community is represented by a list of (<node>, <time instant>)

print(f"{len(dynamic_communities)} dynamic communities have been found")

import importlib
import lago.l_modularity_function as lmf

importlib.reload(lmf)
longitudinal_modularity = lmf.longitudinal_modularity

## Compute Longitudinal Modularity score
## (the higher the better / maximum is 1)
long_mod_score = longitudinal_modularity(
    my_linkstream, 
    dynamic_communities,
    lex_type="MM",
    omega=2
    )

print(f"Longitudinal Modularity score of {long_mod_score} ")

11 dynamic communities have been found
Longitudinal Modularity score of 0.35099 


In [9]:
from __future__ import annotations

from typing import Dict, Hashable, Iterable, Tuple, Any, Optional, Union
import pandas as pd

Node = Hashable
Time = Hashable


def commu_dict_to_interaction_csv(
    commu_dict: Dict[Any, Iterable[Tuple[Node, Time]]],
    syn_csv_path: str = "syn_net.csv",
    out_csv_path: str = "labeled_syn_net.csv",
    *,
    source_col: Union[str, int] = "source",
    destination_col: Union[str, int] = "destination",
    timestamp_col: Union[str, int] = "timestamp",
    syn_header: Optional[int] = "infer",
    syn_sep: str = ",",
    syn_skip_first_row: bool = False,

    node_dtype: Any = int,
    time_dtype: Any = int,

    unknown_commu_value: Any = -1,

    on_conflict: str = "error",
) -> pd.DataFrame:

    syn_df = pd.read_csv(
        syn_csv_path,
        sep=syn_sep,
        header=syn_header,
        skiprows=1 if syn_skip_first_row else None,
        usecols=[source_col, destination_col, timestamp_col],
    ).copy()

    syn_df[source_col] = syn_df[source_col].astype(node_dtype)
    syn_df[destination_col] = syn_df[destination_col].astype(node_dtype)
    syn_df[timestamp_col] = syn_df[timestamp_col].astype(time_dtype)

    active_pairs = set(zip(syn_df[source_col].tolist(), syn_df[timestamp_col].tolist()))
    active_pairs |= set(zip(syn_df[destination_col].tolist(), syn_df[timestamp_col].tolist()))

    node_time_to_commu: Dict[Tuple[Node, Time], Any] = {}

    def _assign(k: Tuple[Node, Time], commu: Any):
        if k not in node_time_to_commu:
            node_time_to_commu[k] = commu
            return
        if node_time_to_commu[k] == commu:
            return
        if on_conflict == "keep_first":
            return
        if on_conflict == "keep_last":
            node_time_to_commu[k] = commu
            return
        raise ValueError(f"Conflict: (node,time)={k} appears in multiple communities: "
                         f"{node_time_to_commu[k]} vs {commu}")

    for commu, pairs in commu_dict.items():
        for u, t in pairs:
            u = node_dtype(u)
            t = time_dtype(t)
            k = (u, t)
            if k in active_pairs:
                _assign(k, commu)

    def _lookup(u: int, t: int):
        return node_time_to_commu.get((u, t), unknown_commu_value)

    syn_df["source_commu"] = [
        _lookup(int(s), int(t)) for s, t in zip(syn_df[source_col], syn_df[timestamp_col])
    ]
    syn_df["destination_commu"] = [
        _lookup(int(d), int(t)) for d, t in zip(syn_df[destination_col], syn_df[timestamp_col])
    ]

    out_df = syn_df.rename(
        columns={
            source_col: "source",
            destination_col: "destination",
            timestamp_col: "timestamp",
        }
    )[["source", "destination", "timestamp", "source_commu", "destination_commu"]]

    out_df.to_csv(out_csv_path, index=False)
    return out_df

real_communities = commu_dict_to_interaction_csv(
    dynamic_communities,
    syn_csv_path="syn_data/p1.0_mu0.2_l5_1.csv",
    out_csv_path="n50_lago.csv",
    syn_header="infer", 
    syn_skip_first_row=False,
    unknown_commu_value=-1, 
    on_conflict="error",
)

(real_communities.head(10))

,source,destination,timestamp,source_commu,destination_commu
0,14,40,0,6,6
1,4,21,0,9,9
2,1,37,1,1,1
3,1,46,2,1,1
4,29,32,4,1,1
5,27,42,9,1,1
6,11,47,11,9,9
7,12,40,13,6,6
8,26,32,14,1,1
9,27,32,15,1,1


In [12]:
from __future__ import annotations

import os
import glob
from typing import Dict, Hashable, Iterable, Tuple, Any, Optional, Union

import pandas as pd
from lago.lago import LinkStream, lago_communities
import lago.l_modularity_function as lmf


Node = Hashable
Time = Hashable


def commu_dict_to_interaction_csv(
    commu_dict: Dict[Any, Iterable[Tuple[Node, Time]]],
    syn_csv_path: str,
    out_csv_path: str,
    *,
    source_col: Union[str, int] = "source",
    destination_col: Union[str, int] = "destination",
    timestamp_col: Union[str, int] = "timestamp",
    syn_header: Optional[int] = "infer",
    syn_sep: str = ",",
    syn_skip_first_row: bool = False,
    node_dtype: Any = int,
    time_dtype: Any = int,
    unknown_commu_value: Any = -1,
    on_conflict: str = "error",
) -> pd.DataFrame:

    syn_df = pd.read_csv(
        syn_csv_path,
        sep=syn_sep,
        header=syn_header,
        skiprows=1 if syn_skip_first_row else None,
        usecols=[source_col, destination_col, timestamp_col],
    ).copy()

    syn_df[source_col] = syn_df[source_col].astype(node_dtype)
    syn_df[destination_col] = syn_df[destination_col].astype(node_dtype)
    syn_df[timestamp_col] = syn_df[timestamp_col].astype(time_dtype)

    active_pairs = set(zip(syn_df[source_col].tolist(), syn_df[timestamp_col].tolist()))
    active_pairs |= set(zip(syn_df[destination_col].tolist(), syn_df[timestamp_col].tolist()))

    node_time_to_commu: Dict[Tuple[Node, Time], Any] = {}

    def _assign(k: Tuple[Node, Time], commu: Any):
        if k not in node_time_to_commu:
            node_time_to_commu[k] = commu
            return
        if node_time_to_commu[k] == commu:
            return
        if on_conflict == "keep_first":
            return
        if on_conflict == "keep_last":
            node_time_to_commu[k] = commu
            return
        raise ValueError(
            f"Conflict: (node,time)={k} appears in multiple communities: "
            f"{node_time_to_commu[k]} vs {commu}"
        )

    for commu, pairs in commu_dict.items():
        for u, t in pairs:
            u = node_dtype(u)
            t = time_dtype(t)
            k = (u, t)
            if k in active_pairs:
                _assign(k, commu)

    def _lookup(u: int, t: int):
        return node_time_to_commu.get((u, t), unknown_commu_value)

    syn_df["source_commu"] = [
        _lookup(int(s), int(t)) for s, t in zip(syn_df[source_col], syn_df[timestamp_col])
    ]
    syn_df["destination_commu"] = [
        _lookup(int(d), int(t)) for d, t in zip(syn_df[destination_col], syn_df[timestamp_col])
    ]

    out_df = syn_df.rename(
        columns={
            source_col: "source",
            destination_col: "destination",
            timestamp_col: "timestamp",
        }
    )[["source", "destination", "timestamp", "source_commu", "destination_commu"]]

    os.makedirs(os.path.dirname(out_csv_path), exist_ok=True)
    out_df.to_csv(out_csv_path, index=False)
    return out_df


INPUT_DIR = "syn_data"
RESULTS_DIR = "results"

csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.csv")))
print(f"Found {len(csv_files)} files in {INPUT_DIR}")

longitudinal_modularity = lmf.longitudinal_modularity

for csv_path in csv_files:
    file_name = os.path.splitext(os.path.basename(csv_path))[0]
    out_dir = os.path.join(RESULTS_DIR, file_name)
    out_csv_path = os.path.join(out_dir, "lago.csv")

    print(f"\nProcessing: {csv_path}")

    df = pd.read_csv(
        csv_path,
        header=None,
        index_col=False,
        names=["source", "destination", "timestamp"],
        skiprows=1,
    )
    time_links = df.values.tolist()

    my_linkstream = LinkStream(is_stream_graph=False)
    my_linkstream.add_links(time_links)

    print(
        f"The link stream consists of {my_linkstream.nb_edges} temporal edges "
        f"(or time links) accross {my_linkstream.nb_nodes} nodes and "
        f"{my_linkstream.network_duration} time steps, of which only "
        f"{my_linkstream.nb_timesteps} contain activity."
    )

    my_linkstream = LinkStream(is_stream_graph=True)
    my_linkstream.add_links(time_links)

    dynamic_communities = lago_communities(
        my_linkstream,
        nb_iter=3,
    )

    print(f"{len(dynamic_communities)} dynamic communities have been found")

    long_mod_score = longitudinal_modularity(
        my_linkstream,
        dynamic_communities,
        lex_type="MM",
        omega=2,
    )

    print(f"Longitudinal Modularity score of {long_mod_score}")

    real_communities = commu_dict_to_interaction_csv(
        dynamic_communities,
        syn_csv_path=csv_path,
        out_csv_path=out_csv_path,
        syn_header="infer",
        syn_skip_first_row=False,
        unknown_commu_value=-1,
        on_conflict="error",
    )

    print(f"Saved: {out_csv_path}")
    print(real_communities.head())

Found 500 files in syn_data

Processing: syn_data/p0.85_mu0.05_l10_1.csv
The link stream consists of 2485 temporal edges (or time links) accross 50 nodes and 2452 time steps, of which only 1551 contain activity.


/var/folders/g1/wg9xf7ks75b49ljdm6l5zy280000gp/T/ipykernel_4714/1548222649.py:112: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(


29 dynamic communities have been found
Longitudinal Modularity score of 0.4836
Saved: results/p0.85_mu0.05_l10_1/lago.csv
   source  destination  timestamp  source_commu  destination_commu
0      34           45          0             8                  8
1      13           34          5             8                  8
2      14           34          6             8                  8
3      12           33          7             8                  8
4       4           47          7            21                 21

Processing: syn_data/p0.85_mu0.05_l10_2.csv
The link stream consists of 2485 temporal edges (or time links) accross 50 nodes and 2452 time steps, of which only 1551 contain activity.


/var/folders/g1/wg9xf7ks75b49ljdm6l5zy280000gp/T/ipykernel_4714/1548222649.py:112: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(


32 dynamic communities have been found
Longitudinal Modularity score of 0.48972
Saved: results/p0.85_mu0.05_l10_2/lago.csv
   source  destination  timestamp  source_commu  destination_commu
0      36           38          0            25                 25
1      13           47          5            25                 25
2      15           24          6            26                 26
3      12           42          7            25                 25
4       5           39          7            26                 26

Processing: syn_data/p0.85_mu0.05_l10_3.csv
The link stream consists of 2485 temporal edges (or time links) accross 50 nodes and 2452 time steps, of which only 1551 contain activity.


...

31 dynamic communities have been found
Longitudinal Modularity score of 0.26047
Saved: results/p1.0_mu0.2_l5_3/lago.csv
   source  destination  timestamp  source_commu  destination_commu
0       7           16          5            18                 18
1      42           46          6            19                 19
2       2           29          7            18                 18
3       7           32          7            18                 18
4       3           10          8            16                 16

Processing: syn_data/p1.0_mu0.2_l5_4.csv
The link stream consists of 1276 temporal edges (or time links) accross 50 nodes and 1258 time steps, of which only 797 contain activity.


/var/folders/g1/wg9xf7ks75b49ljdm6l5zy280000gp/T/ipykernel_4714/1548222649.py:112: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(


33 dynamic communities have been found
Longitudinal Modularity score of 0.21448
Saved: results/p1.0_mu0.2_l5_4/lago.csv
   source  destination  timestamp  source_commu  destination_commu
0       7           32          5             7                  7
1      38           43          6            16                 16
2       3           17          7            16                 16
3       8            9          7            16                 16
4       4           31          8            16                 16

Processing: syn_data/p1.0_mu0.2_l5_5.csv
The link stream consists of 1276 temporal edges (or time links) accross 50 nodes and 1258 time steps, of which only 797 contain activity.


/var/folders/g1/wg9xf7ks75b49ljdm6l5zy280000gp/T/ipykernel_4714/1548222649.py:112: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(


29 dynamic communities have been found
Longitudinal Modularity score of 0.30882
Saved: results/p1.0_mu0.2_l5_5/lago.csv
   source  destination  timestamp  source_commu  destination_commu
0       6           31          5            14                 14
1      36           45          6             3                  3
2       3            7          7            14                 14
3       6           48          7            14                 14
4       2           33          8            27                 27
